# Day 03 - CICIDS2017 Data Exploration

Goal: validate `data/raw/sample.csv`, understand labels/features, and record decisions for Day 05 preprocessing.

## Setup

Run `scripts/create_dataset_sample.py` first if `data/raw/sample.csv` does not exist.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SAMPLE_PATH_CANDIDATES = [Path("data/raw/sample.csv"), Path("../data/raw/sample.csv")]
SAMPLE_PATH = next((path for path in SAMPLE_PATH_CANDIDATES if path.exists()), SAMPLE_PATH_CANDIDATES[0])
SAMPLE_PATH

In [ ]:
if not SAMPLE_PATH.exists():
    raise FileNotFoundError(
        "Missing data/raw/sample.csv. Put CICIDS2017 CSV files in data/raw/MachineLearningCVE/ and run: "
        "python scripts/create_dataset_sample.py data/raw/<source-file>.csv --output data/raw/sample.csv"
    )

df = pd.read_csv(SAMPLE_PATH)
df.columns = df.columns.str.strip()
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

## Columns and Label

In [ ]:
pd.Series(df.columns, name="columns")

In [ ]:
LABEL_COLUMN = "Label"

label_counts = df[LABEL_COLUMN].value_counts(dropna=False)
binary_labels = df[LABEL_COLUMN].astype(str).str.strip().str.upper().ne("BENIGN").astype(int)
binary_counts = binary_labels.value_counts().rename(index={0: "normal", 1: "anomaly"})

display(label_counts)
display(binary_counts)

## Data Quality Checks

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(30)

In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_rows

In [ ]:
numeric_df = df.select_dtypes(include=["number"])
numeric_df.describe(percentiles=[0.5, 0.95, 0.99]).T.sort_index().head(40)

## Logical Schema Mapping

This cell checks which planned MVP fields can be created from the selected CICIDS2017 sample.

In [ ]:
column_candidates = {
    "timestamp": ["Timestamp"],
    "src_ip": ["Source IP", "Src IP"],
    "dst_ip": ["Destination IP", "Dst IP"],
    "src_port": ["Source Port", "Src Port"],
    "dst_port": ["Destination Port", "Dst Port"],
    "protocol": ["Protocol"],
    "duration": ["Flow Duration"],
    "fwd_packets": ["Total Fwd Packets", "Tot Fwd Pkts"],
    "bwd_packets": ["Total Backward Packets", "Tot Bwd Pkts"],
    "fwd_bytes": ["Total Length of Fwd Packets", "TotLen Fwd Pkts"],
    "bwd_bytes": ["Total Length of Bwd Packets", "TotLen Bwd Pkts"],
    "label": ["Label"],
}

resolved = {}
for logical_name, candidates in column_candidates.items():
    resolved[logical_name] = next((candidate for candidate in candidates if candidate in df.columns), None)

pd.Series(resolved, name="source_column")

## Quick Derived Features

These are prototype calculations for Day 05. Keep the production version in `pipelines/preprocess.py` later.

In [ ]:
work = df.copy()

fwd_packets = resolved.get("fwd_packets")
bwd_packets = resolved.get("bwd_packets")
fwd_bytes = resolved.get("fwd_bytes")
bwd_bytes = resolved.get("bwd_bytes")
duration = resolved.get("duration")

if fwd_packets and bwd_packets:
    work["packets"] = pd.to_numeric(work[fwd_packets], errors="coerce") + pd.to_numeric(work[bwd_packets], errors="coerce")
if fwd_bytes and bwd_bytes:
    work["bytes"] = pd.to_numeric(work[fwd_bytes], errors="coerce") + pd.to_numeric(work[bwd_bytes], errors="coerce")
if duration:
    work["duration"] = pd.to_numeric(work[duration], errors="coerce")

safe_packets = work.get("packets", pd.Series(np.nan, index=work.index)).replace(0, np.nan)
safe_duration = work.get("duration", pd.Series(np.nan, index=work.index)).replace(0, np.nan)

work["binary_label"] = work[LABEL_COLUMN].astype(str).str.strip().str.upper().ne("BENIGN").astype(int)
work["bytes_per_packet"] = work.get("bytes", np.nan) / safe_packets
work["packets_per_second"] = work.get("packets", np.nan) / safe_duration
work["bytes_per_second"] = work.get("bytes", np.nan) / safe_duration

work[["binary_label", "bytes_per_packet", "packets_per_second", "bytes_per_second"]].describe().T

## Day 03 Conclusion

Summary from the generated `data/raw/sample.csv`:

- Dataset usable for MVP: yes, for binary flow-based anomaly detection.
- Source CSV folder used: `data/raw/MachineLearningCVE/`.
- Sample rows: 50,000.
- Sample columns: 79.
- Normal/anomaly ratio: 25,000 / 25,000.
- Missing cells: 36.
- Duplicate rows: 2,945.
- Columns missing from target logical schema: timestamp, source IP, destination IP, source port, protocol.
- Dashboard adjustment: use destination port, attack label distribution, packet/byte/rate features, and anomaly counts instead of top IP or protocol distribution for the first MVP.
- Columns to clean/drop on Day 05: duplicate/constant columns, missing or infinite rate values, repeated header-length columns if confirmed.
- Main risks: class distribution in the full dataset is imbalanced, and rare attack classes are tiny after balanced binary sampling.